# Figure 10 — Griffing distance-based partition recovery under subsampling

Distance-matrix analogue of Figure 3 (Fiedler recovery on synthetic CBM).
Pipeline:

1. Generate a tree + sequences → JC distance matrix $D \in \mathbb R^{m\times m}$.
2. Mean-center: $B = (I - \mathbf{11}^\top/m)\, D\, (I - \mathbf{11}^\top/m)$.
3. Reference partition vector: $v_{\text{ref}} =$ eigenvector of $B$ with the
   largest $|\lambda|$ (classical MDS first coordinate). The sign pattern of
   $v_{\text{ref}}$ is Griffing's terminal-node partition.
4. For each $(p, \text{rep})$ sub-sample $D \to \hat D$ (uniform IPW,
   zero-fill unsampled), compute $\hat v$ = leading eigvec of $J\hat D J$,
   and record sign-agnostic agreement with $v_{\text{ref}}$.

Why we expect this to behave differently from figures 8/9: under IPW
+ mean-imputation $\hat D \approx D/p$ uniformly, so $\hat B \approx B/p$
and **the leading eigenvector is invariant under positive scaling**. The
sign pattern should survive a huge spec-norm error — the question is
whether the *stochastic* component of $\hat D$ (which shrinks with $n$)
is enough to flip individual signs.

Two sweeps (one per tree model): **birth_death** (n-comparable baseline,
same as figure 9) and **kingman** (failed for NJ because tip branches
shrink as $O(1/n^2)$ — but Griffing only needs the dominant split, not
all internal branches).

In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys, json
from datetime import datetime
from pathlib import Path
from IPython.display import Image, display

_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..', '..'))
if _root not in sys.path:
    sys.path.insert(0, _root)

for _mod in [m for m in list(sys.modules) if m.startswith(('scripts.', 'src.'))]:
    del sys.modules[_mod]

from src.config.presets import custom_config
from src.runners.griffing_sweep import griffing_sweep_for_params
from src.utils.griffing_io import load_griffing_results
from scripts.plot_griffing_overlay import _plot_overlay, _compute_p_stars, RECOVERY_THRESHOLDS

## Choose: load or launch (per tree model)
Set `RUN_DIRS[tree_model]` to an existing sweep directory to **load**. Set to `None` to launch a fresh sweep with the grid below.

In [ ]:
# Default: launch fresh sweeps. After the first run, replace `None`
# with the produced Path(...) so re-runs just reload + plot.
RUN_DIRS = {
    'birth_death': None,
    'kingman':     None,
}

TAXA   = [128, 256, 512, 1024, 2048]
SEQLEN = 10000
REPS   = 5
P_VALS = [1.0, 0.9, 0.5, 0.1, 0.05, 0.01, 0.005, 0.001, 0.0005, 0.0001]
IMPUTATION = 'zero'

TREE_KWARGS = {
    'birth_death': dict(birth_rate=1.0, death_rate=0.0),
    'kingman':     dict(pop_size=1.0),
}

for tree_model in list(RUN_DIRS.keys()):
    rd = RUN_DIRS[tree_model]
    if rd is None:
        rd = Path(_root) / 'results' / 'runs' / (datetime.now().strftime('%Y%m%d-%H%M%S')
                                         + f'-griffing_sweep_{tree_model}_L{SEQLEN}')
        for n in TAXA:
            cfg = custom_config(num_taxa=n, sequence_length=SEQLEN, mutation_rate=0.1,
                                tree_model=tree_model, seq_model='JC69',
                                p_values=P_VALS, bootstrap_reps=REPS,
                                sampling_method='uniform', matrix_kind='distance',
                                **TREE_KWARGS[tree_model])
            griffing_sweep_for_params(cfg, n, SEQLEN, str(rd / f'n{n}_L{SEQLEN}'),
                                       imputation=IMPUTATION)
        RUN_DIRS[tree_model] = rd
    else:
        RUN_DIRS[tree_model] = Path(rd)
    print(f"{tree_model:>12s}: {RUN_DIRS[tree_model]}")

## Overlay + $p^*$ table per tree model

In [ ]:
for tree_model, rd in RUN_DIRS.items():
    print(f"\n=== {tree_model} (sweep dir: {rd}) ===")
    rows = _compute_p_stars(rd)
    header = f"{'n':>5}  " + '  '.join(f">={int(t*100)}%" for t in RECOVERY_THRESHOLDS)
    print(header)
    for r in rows:
        cells = [f"{r['n']:>5}"]
        for t in RECOVERY_THRESHOLDS:
            p = r['p_star_by_threshold'].get(t)
            cells.append(f"{p:>5.3f}" if (p is not None) else '  n/a')
        print('  '.join(cells))

    out_path = rd / 'griffing_overlay.png'
    _plot_overlay(rd, out_path)
    display(Image(filename=str(out_path)))